In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
create schema if not exists workspace.gold

In [0]:
spark.read.table('workspace.silver.silver_products').printSchema()

In [0]:
%sql
show tables in workspace.silver

In [0]:
df_products=spark.read.table('workspace.silver.silver_products')
df_aisle=spark.read.table('workspace.silver.silver_aisles')
df_department=spark.read.table('workspace.silver.silver_departments')

In [0]:
df_products.printSchema()

In [0]:
df_aisle.printSchema()

In [0]:
df_department.printSchema()

In [0]:
print('count of products : ',df_products.count())
print('count of aisles : ',df_aisle.count())
print('count of departments : ',df_department.count())

In [0]:
from pyspark.sql.functions import broadcast,col

In [0]:
dim_products=df_products.alias('p').join(broadcast(df_aisle.alias('a')),on='aisle_id',how='left')\
    .join(broadcast(df_department.alias('d')),df_products['department_id']==df_department['department_id'],'left')\
        .select(
            col("p.product_id"),
            col("p.product_name"),
            col("p.aisle_id"),
            col('a.aisle'),
            col('p.department_id'),
            col('d.department')
                
         )
    

In [0]:
dim_products.count()

In [0]:
dim_products.write.format('delta')\
    .mode('overwrite')\
    .saveAsTable('workspace.gold.dim_products')

In [0]:
%sql
show tables in workspace.gold

In [0]:
dim_user=spark.read.table('workspace.silver.silver_orders')\
    .select('user_id').distinct()

In [0]:
dim_user.write.format('delta')\
    .mode('overwrite')\
    .saveAsTable('workspace.gold.dim_user')


In [0]:
df_items = spark.table("workspace.silver.silver_order_products_mix")
df_orders = spark.table("workspace.silver.silver_orders")

order_metrics = (
    df_items
    .groupBy("order_id")
    .agg(
        count("product_id").alias("total_items_in_cart"),
        sum("reordered").alias("total_reordered_items")
    )
)




In [0]:
fact_orders = (
    df_orders.alias("o")
    .join(order_metrics.alias('om'), on="order_id", how="inner")
    .select(
        col("order_id"),
        col("o.user_id"),
        col("o.order_number"),
        col("o.order_dow"),
        col("o.order_hour_of_day"),
        col("o.days_since_prior_order"),
        col("o.eval_set").alias('order_type'),
        col("om.total_items_in_cart"),
        col("om.total_reordered_items")
    )
)

In [0]:
fact_orders.write.format('delta')\
    .mode('overwrite')\
    .saveAsTable('workspace.gold.fact_orders')

In [0]:
fact_orders.count()

In [0]:
%sql
OPTIMIZE workspace.gold.fact_orders
ZORDER BY (user_id);

In [0]:
fact_order_items = (
    df_items.join(df_orders, on="order_id", how="inner")
    .select(
        col("order_id"),
        col("user_id"),
        col("product_id"),
        col("order_number"),
        col("order_dow"),
        col("order_hour_of_day"),
        col("days_since_prior_order"),
        col("add_to_cart_order"),
        col("reordered"),
        df_items["order_type"]
    )
)

# Save to Gold
fact_order_items.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fact_order_items")

In [0]:
%sql
OPTIMIZE workspace.gold.fact_order_items
ZORDER BY (user_id,product_id);

In [0]:
%sql
SHOW TABLES IN workspace.gold;